# Day 27: Input Validation and Reliability

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Centralize strict validation and saved-model predictions for all serving paths.

## Theory

Rejecting malformed requests before inference prevents silent schema drift and target leakage. Numeric fields are finite, integral and within documented ranges; categorical values use explicit domains.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Load real reference inputs and the shared service.

In [2]:
from src.dataset import original_split
from src.prediction import predict,model_id
X=original_split()[1]
print("Model version:",model_id())
print(predict(X.head(3)).to_string(index=False))

Model version: 1bedf35d0d56c4f0
predicted_category  model_estimated_proba_High  model_estimated_proba_Low  model_estimated_proba_Medium
            Medium                    0.000000                   0.090909                      0.909091
               Low                    0.000000                   1.000000                      0.000000
            Medium                    0.333333                   0.000000                      0.666667


Reject a target column rather than dropping it silently.

In [3]:
from src.validation import InputValidationError
bad=X.head(1).assign(G3=20)
try:
    predict(bad)
except InputValidationError as e:
    print("Expected rejection:",e)
else:
    raise AssertionError("Target column accepted")

Expected rejection: Missing fields: []; unexpected fields: ['G3']


Run the reliability regression tests.

In [4]:
import subprocess,sys
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_prediction.py"],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0,r.stderr

..............                                                           [100%]
14 passed in 1.69s



## Findings and interpretation

All serving outputs use the saved model and retain its class mapping. The regression suite checks invalid numbers, unknown categories, missing/extra columns, empty/large batches and order invariance.

## Limitations

The bounded schema intentionally rejects out-of-domain students. This validation checks data structure, not whether submitted attributes are true.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Centralize strict validation and saved-model predictions for all serving paths.

## Next-day plan

Add reusable row-level data-quality diagnostics.